### Module 3: Scaled Dot-Product Attention

### The concept

This is the core operation of the entire transformer. Every other module is essentially scaffolding around this. The idea is simple: **each token asks a question, and every other token answers it** — the response is a weighted sum of values, where the weights come from how well each answer matches the question.

Formally, each token is projected into three vectors:

$$Q = XW_Q, \quad K = XW_K, \quad V = XW_V$$

where $X \in \mathbb{R}^{B \times N \times D}$ and each weight matrix $W_Q, W_K, W_V \in \mathbb{R}^{D \times d_k}$ with $d_k = D / h$ for $h$ attention heads.

Then attention is computed as:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

Breaking this down step by step with shapes (single head, single batch for clarity):

$$\underbrace{Q}_{N \times d_k} \underbrace{K^\top}_{d_k \times N} \rightarrow \underbrace{A}_{N \times N} \xrightarrow{\div\sqrt{d_k}} \xrightarrow{\text{softmax}} \underbrace{\hat{A}}_{N \times N} \xrightarrow{\times V} \underbrace{N \times d_k}$$

$\hat{A}_{ij}$ is the attention weight — how much token $i$ attends to token $j$.

With our tiny model: $D=16$, $h=2$ heads, so $d_k = 8$ per head.

---



### 🤔 Pre-coding questions

**Q1.** The attention matrix $A \in \mathbb{R}^{N \times N}$ has shape $16 \times 16$ in our model. What does the entry $A_{ij}$ represent geometrically? And what does an entire **row** $A_{i,:}$ represent?

- Answer: Doesn't the Aij somehow look like the row-col pairs to which each token is most associated to? For example the token 0 and token 1 relationship is in A[0,1]. One row shows the relationship of token 0 with itself and all other tokens! A slight note:

$$A_{ij} = Q_i \cdot K_j^\top \quad \text{but} \quad A_{ji} = Q_j \cdot K_i^\top$$

Since $W_Q \neq W_K$, we have $Q_i \neq K_i$ in general, so $A_{ij} \neq A_{ji}$. The matrix is **asymmetric**. Row $i$ means *"token $i$ as a query, scoring all keys"* — how much $i$ wants to look at others. Column $j$ means *"token $j$ as a key, being looked at by all queries"* — how much others find $j$ relevant. These are genuinely different roles.

**Q2.** Why do we divide by $\sqrt{d_k}$ before the softmax? What goes wrong without it — and does it matter more when $d_k$ is large or small?

- Answer: The dot product $Q_i \cdot K_j$​ has variance that scales with $d_k$​. When $d_k$​ is large, the raw scores grow proportionally to $\sqrt{d_k}$​, pushing softmax toward saturation. Dividing by $\sqrt{d_k}$​ normalizes the variance back to 1 regardless of head size. So the scaling matters more as $d_k$​ grows.

**Q3.** After softmax, each row of $\hat{A}$ sums to 1. So the output for token $i$ is a **convex combination** of all value vectors. What does it mean when $\hat{A}_{ij} \approx 1$ for a single $j$ and $\approx 0$ everywhere else? What about when all entries in row $i$ are equal at $\frac{1}{N}$?

- Answer: If A_{ij} is close to 1, then that means that the score pair ij matches closely together or has a strong relation to itself. If all entries are equally scored (i.e., 1/N) then that means there is no prevalent information to say to which the token is most closest to. To put simply:

- $\hat{A}_{ij} \approx 1$: token $i$ almost entirely copies token $j$'s value vector — hard attention on one token
- $\hat{A}_{i,:} = \frac{1}{N}$​: token $i$ takes a uniform average of all value vectors — effectively ignoring all positional structure

**Q4.** In multi-head attention we run $h=2$ independent attention operations in parallel, each with its own $W_Q^{(h)}, W_K^{(h)}, W_V^{(h)}$ and $d_k = D/h = 8$. After computing attention for each head, we concatenate the outputs and project back to $D$. Why not just run one big attention with the full $D$ dimensions instead of splitting into heads?

- Answer: That's an interesting question. I don't know the exact answer but my intuition says we want to distribute the information across the entire dimension embedded D. In a sense it's like making a "voting" method where we have different heads to have different opinions. And each head would predict something different. In a more concrete saying:
- Each head learns to attend to a different relational pattern simultaneously. In a ViT, one head might specialize in local spatial adjacency (nearby patches), another in global semantic similarity (similar textures anywhere in the image). A single full-$D$ attention head would have to collapse all these patterns into one — heads allow them to coexist in parallel without interfering.

**Q5.** The $Q$, $K$, $V$ projections use **different** weight matrices. $Q$ and $K$ interact to produce attention weights, while $V$ is what actually gets aggregated. What would happen if you set $W_Q = W_K = W_V$ — same weights for all three?

- Answer: results in 3 things:
- $A$ becomes symmetric ($A_{ij} = A_{ji}$​) since $Q = K$ — you lose the asymmetric query/key distinction from Q1
- The model loses the ability to separately learn *"what to search for"* ($Q, K$) vs *"what to return"* ($V$) — these are genuinely different tasks collapsed into one projection
- In practice, gradients for all three matrices become entangled and training becomes much harder.

**Q6.** After softmax, the output for token $i$ is $\hat{A}_{i,:} \cdot V$. If patch $i$ is in the middle of the image, what kind of attention pattern would you intuitively expect $\hat{A}_{i,:}$​ to look like after training — and why?

For a middle patch after training, the more likely pattern is **high scores toward its spatial neighbors and semantically similar patches elsewhere** — it's reaching *outward* to gather context, not pointing back at itself. Self-attention score (A_{ii}​) tends to be moderate, not dominant.

# Coding Exercise

In [2]:
import numpy as np

# ── Tiny model constants (same as before) ─────────────────────
B, N, D = 2, 16, 16
h        = 2          # attention heads
d_k      = D // h     # 8 per head

# Ideally this should come from mod2
z_pe = np.random.randn(B, N, D) * 0.02   # (2, 16, 16)

# Input: position-aware token embeddings from Module 2
# shape: (B, N, D)
X = z_pe.copy()

# ── Weight matrices (one set per head) ────────────────────────
np.random.seed(42)
WQ = np.random.randn(h, D, d_k) * 0.02   # (2, 16, 8)
WK = np.random.randn(h, D, d_k) * 0.02
WV = np.random.randn(h, D, d_k) * 0.02
WO = np.random.randn(h * d_k, D) * 0.02  # output projection (16, 16)


# ─────────────────────────────────────────────────────────────
# STAGE 1: Single-head attention
# ─────────────────────────────────────────────────────────────

def softmax(x, axis=-1):
    """
    Numerically stable softmax along a given axis.
    Hint: subtract the max before exponentiating.
    """
    x_max = np.max(x, axis=axis, keepdims=True)   # subtract max for stability
    e_x   = np.exp(x - x_max)                     # numerator
    return e_x / np.sum(e_x, axis=axis, keepdims=True)  # normalize

def single_head_attention(X, Wq, Wk, Wv):
    """
    Single head scaled dot-product attention.

    Input:  X   shape (B, N, D)
            Wq  shape (D, d_k)
            Wk  shape (D, d_k)
            Wv  shape (D, d_k)
    Output: shape (B, N, d_k)

    Steps:
        1. Project X into Q, K, V
        2. Compute raw attention scores QK^T
        3. Scale by 1/sqrt(d_k)
        4. Softmax over the key dimension (axis=-1)
        5. Weighted sum over V
    """
    Q = X @ Wq                              # (B, N, d_k)
    K = X @ Wk                              # (B, N, d_k)
    V = X @ Wv                              # (B, N, d_k)
    A = Q @ K.transpose(0, 2, 1)            # (B, N, N)
    A_hat = softmax(A / np.sqrt(d_k))       # (B, N, N)
    out = A_hat @ V                         # (B, N, d_k)
    return out


# ── Stage 1 shape check ───────────────────────────────────────
out_single = single_head_attention(X, WQ[0], WK[0], WV[0])
assert out_single.shape == (B, N, d_k), f"Got {out_single.shape}"
print("single head out:", out_single.shape)   # expect (2, 16, 8)


# ─────────────────────────────────────────────────────────────
# STAGE 2: Multi-head attention
# ─────────────────────────────────────────────────────────────

def multi_head_attention(X, WQ, WK, WV, WO):
    """
    Multi-head attention.

    Input:  X    shape (B, N, D)
            WQ   shape (h, D, d_k)
            WK   shape (h, D, d_k)
            WV   shape (h, D, d_k)
            WO   shape (h*d_k, D)
    Output: shape (B, N, D)

    Steps:
        1. Run single_head_attention for each head
        2. Concatenate head outputs along last axis → (B, N, h*d_k)
        3. Project through WO → (B, N, D)
    """
    head_outputs = []
    for i in range(h):
        head_out = single_head_attention(X, WQ[i], WK[i], WV[i])  # (B, N, d_k)
        head_outputs.append(head_out)
    concat_heads = np.concatenate(head_outputs, axis=-1)  # (B, N, h*d_k)
    out = concat_heads @ WO                               # (B, N, D)
    return out


# ── Stage 2 shape check ───────────────────────────────────────
out_mha = multi_head_attention(X, WQ, WK, WV, WO)
assert out_mha.shape == (B, N, D), f"Got {out_mha.shape}"
print("multi head out:", out_mha.shape)   # expect (2, 16, 16)


# ── Sanity check: inspect one attention map ───────────────────
# After you get Stage 1 working, add this to visualise the
# attention weights for batch item 0, head 0
# A_hat should be shape (N, N) = (16, 16)
# Each row should sum to 1.0
def get_attention_map(X, Wq, Wk):
    Q = X @ Wq                              # (B, N, d_k)
    K = X @ Wk                              # (B, N, d_k)
    A = Q @ K.transpose(0, 2, 1)           # (B, N, N)
    A_hat = softmax(A / np.sqrt(d_k))
    return A_hat

A_hat = get_attention_map(X, WQ[0], WK[0])
print("Row sums (should all be 1.0):", A_hat[0].sum(axis=-1).round(4))

single head out: (2, 16, 8)
multi head out: (2, 16, 16)
Row sums (should all be 1.0): [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


# Special Notes
### Is $h$ a hyperparameter? 

Yes, with one hard constraint: $D$ must be divisible by $h$ so that $d_k = D/h$ is a whole number. Common choices in practice:
- ViT Small, $D = 384$, $h = 6$, $d_k = 64$
- ViT Base, $D = 768$, $h = 12$, $d_k = 64$
- ViT Large, $D = 1024$, $h = 16$, $d_k = 64$

### Does more heads mean more information?
No — total capacity stays fixed at $D$. More heads means the **same $D$ dimensions get carved into more specialized subspaces**, each capturing a different relationship pattern. But each head gets *less* capacity ($d_k$​ shrinks). The tradeoff is:

- Fewer heads, larger $d_k$​: each head is more expressive but can only capture one broad pattern
- More heads, smaller $d_k$​: more diverse patterns in parallel, but each head has less room to work with

Beyond a certain point, adding heads hurts because $d_k$​ becomes too small to represent anything meaningful. That's why $d_k = 64$ became a practical sweet spot.

### Recall how to compute a numerically stable softmax:

The equation: 

$$\text{softmax}(\mathbf{x})_i = \frac{e^{x_i}}{\sum_{j} e^{x_j}}$$

The numerical stability problem: If any $x_i$​ is large (e.g. 1000), $e^{1000}$ overflows to inf. The trick is to subtract the max first — it doesn't change the result mathematically:

$$\text{softmax}(\mathbf{x})_i = \frac{e^{x_i - \max(\mathbf{x})}}{\sum_{j} e^{x_j - \max(\mathbf{x})}}$$

because the $e^{-\max(x)}$ cancels in numerator and denominator.